# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR² clinical oncology dataset using the `mlcroissant` library.

### Dataset Source
The dataset schema is defined by a Croissant JSON-LD hosted at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset's metadata and explore dataset-level information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, not a dict/list

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.date_published}")
print(f"License: {metadata.license}")

## 2. Data Overview
List all available record sets (tables), their fields, and their `@id`. Only references by `@id` are used.

In [ ]:
# Enumerate all record sets defined in the dataset
print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"- Name: {record_set.name} | @id: {record_set.id}")
    # List fields for each record set
    for field in record_set.fields:
        print(f"    - Field: {field.name} | @id: {field.id} | Type: {field.data_type}")

## 3. Data Extraction
We'll load all available record sets into pandas DataFrames using their `@id`. Use the record set and field `@id`s found above for future references.

In [ ]:
# Prepare DataFrames for each record set by @id
record_sets_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_id} | Shape: {df.shape}")

# For demonstration, pick the main clinical tabular data record set by @id (first one for example)
main_record_set_id = record_sets_ids[0] if record_sets_ids else None
if main_record_set_id is not None:
    print(f"\nMain record set DataFrame columns (@id):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric column (by field `@id`), apply a filter, normalize, and optionally group by a categorical field, all using `@id`s.

In [ ]:
# Identify a numeric field (by @id) in the main record set
main_df = dataframes[main_record_set_id]
# Display columns for field selection
print("Available columns in selected record set:")
for col in main_df.columns:
    print(f"- {col}")

# Suppose 'schema:age' is the @id for age (update if another field represents this with a different @id)
numeric_field_id = 'schema:age'  # Update to actual @id from the field overview if necessary

# Let's choose a threshold value for demonstration
threshold = 60

if numeric_field_id in main_df.columns:
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered rows (where {numeric_field_id} > {threshold}): {filtered_df.shape[0]} rows")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field, e.g., 'schema:sex' (update with the correct @id)
    group_field_id = 'schema:sex'  # Update as needed
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df)
else:
    print(f"Column {numeric_field_id} not found in main record set. Please update 'numeric_field_id' to an actual numeric @id.")

## 5. Visualization
Let's visualize the numeric field distribution and the group-wise averages using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of numeric field
if numeric_field_id in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Barplot of means grouped by categorical field (if available)
if 'grouped_df' in locals():
    plt.figure(figsize=(6, 4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion

- The FAIR² clinical dataset was loaded and explored using the `mlcroissant` library.
- We identified and referenced all dataset components explicitly by their `@id` (record sets, fields).
- The EDA process included filtering, normalization, and group-wise summaries, all performed via `@id` references.
- Data visualizations provided further insights into the distribution and structure of key clinical variables.

**Next steps:** Apply domain modeling, more advanced analyses, or machine learning workflows using the loaded DataFrames.